In [1]:
%cd /kaggle/working

!git clone https://github.com/phusroyal/ViHOS.git
%cd /kaggle/working/ViHOS

!find . -maxdepth 3 -type f | sed 's#^\./##' | sort | head -100

/kaggle/working
Cloning into 'ViHOS'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 152 (delta 65), reused 102 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 5.80 MiB | 19.66 MiB/s, done.
Resolving deltas: 100% (65/65), done.
/kaggle/working/ViHOS
codes/Code to tokenize data/Tokenize by Word.ipynb
codes/Code to tokenize data/Tokenizer by Syllabel.ipynb
codes/Models/BiLSTM-CRF _HOS_.ipynb
codes/Models/PhoBERT_HOS.ipynb
codes/Models/Where to download word2vec models?
codes/Models/XLMR_HOS.ipynb
codes/Requirement.txt
data/README.md
data/Span_Extraction_based_version/dev.csv
data/Span_Extraction_based_version/train.csv
data/Test_data/test.csv
.git/config
.git/description
.git/HEAD
.git/hooks/applypatch-msg.sample
.git/hooks/commit-msg.sample
.git/hooks/fsmonitor-watchman.sample
.git/hooks/post-update.sample
.git/hooks/pre-applypatch.sample
.git/hooks/pre-commi

In [2]:
from sklearn.metrics import f1_score
import numpy as np
import os
from tqdm import tqdm

In [3]:
import tensorflow as tf

tf.keras.backend.clear_session()

# clear gpu memory using torch
import torch
torch.cuda.empty_cache()

# clear output
from IPython.display import clear_output
clear_output()

In [4]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [5]:
REPO_DIR = "/kaggle/working/ViHOS"
DATA_DIR = f"{REPO_DIR}/data"

train_path = f"{DATA_DIR}/Sequence_labeling_based_version/Word/train_BIO_Word.csv"
dev_path   = f"{DATA_DIR}/Sequence_labeling_based_version/Word/dev_BIO_Word.csv"

test_bio_path = f"{DATA_DIR}/Sequence_labeling_based_version/Word/test_BIO_Word.csv"
test_gold_path = f"{DATA_DIR}/Test_data/test.csv"
test_path = test_bio_path

In [6]:
from transformers import (
    AutoModel, AutoConfig, XLMRobertaModel,
    AutoTokenizer, AutoModelForSequenceClassification
)

input_model = XLMRobertaModel.from_pretrained("vinai/phobert-large")
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-large")
input_model.resize_token_embeddings(len(tokenizer))

clear_output()

# Data

In [7]:
import pandas as pd
import transformers
import torch
import torch.nn as nn
import pandas as pd

#clear output
from IPython.display import clear_output
clear_output()

In [8]:
def prepare_data(file_path):
    df = pd.read_csv(file_path)

    # Xóa cột Unnamed nếu có
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    # Chuẩn hóa tên cột: Word -> word, Tag -> tag
    df.columns = [c.strip().lower() for c in df.columns]

    required_cols = {"word", "tag"}
    if not required_cols.issubset(set(df.columns)):
        raise ValueError(
            f"File {file_path} thiếu cột word/tag. "
            f"Cột hiện có: {df.columns.tolist()}"
        )

    sentences = []
    labels = []

    if "sentence_id" in df.columns:
        for _, group in df.groupby("sentence_id", sort=False):
            words = group["word"].astype(str).tolist()
            tags = group["tag"].astype(str).tolist()
            sentences.append(words)
            labels.append(tags)
    else:
        if "index" not in df.columns:
            raise ValueError(
                f"File {file_path} không có sentence_id hoặc index. "
                f"Cột hiện có: {df.columns.tolist()}"
            )

        cur_words = []
        cur_tags = []

        for _, row in df.iterrows():
            idx = row["index"]

            if idx == 0 and len(cur_words) > 0:
                sentences.append(cur_words)
                labels.append(cur_tags)
                cur_words = []
                cur_tags = []

            cur_words.append(str(row["word"]))
            cur_tags.append(str(row["tag"]))

        if len(cur_words) > 0:
            sentences.append(cur_words)
            labels.append(cur_tags)

    return sentences, labels

In [9]:
from torch.utils.data import Dataset, DataLoader

label2id = {
    "O": 0,
    "B-HOS": 1,
    "I-HOS": 2,
    "B-T": 1,
    "I-T": 2,
}

id2label = {
    0: "O",
    1: "B-HOS",
    2: "I-HOS",
}


class PhoBERTDataset(Dataset):
    def __init__(self, sentences, tags, tokenizer, max_len=64):
        self.sentences = sentences
        self.tags = tags
        self.tokenizer = tokenizer
        self.max_len = max_len

        self.pad_token_id = tokenizer.pad_token_id
        self.cls_token_id = tokenizer.cls_token_id
        self.sep_token_id = tokenizer.sep_token_id

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        words = self.sentences[idx]
        tags = self.tags[idx]

        input_ids = []
        label_ids = []

        # CLS
        input_ids.append(self.cls_token_id)
        label_ids.append(-100)

        for word, tag in zip(words, tags):
            if tag not in label2id:
                raise ValueError(f"Unknown tag: {tag}")

            sub_tokens = self.tokenizer.tokenize(str(word))

            if len(sub_tokens) == 0:
                continue

            sub_token_ids = self.tokenizer.convert_tokens_to_ids(sub_tokens)

            for j, sub_id in enumerate(sub_token_ids):
                input_ids.append(sub_id)

                # Chỉ tính loss ở subword đầu tiên
                if j == 0:
                    label_ids.append(label2id[tag])
                else:
                    label_ids.append(-100)

        # SEP
        input_ids.append(self.sep_token_id)
        label_ids.append(-100)

        # Truncate
        input_ids = input_ids[:self.max_len]
        label_ids = label_ids[:self.max_len]

        attention_mask = [1] * len(input_ids)

        # Padding
        pad_len = self.max_len - len(input_ids)

        if pad_len > 0:
            input_ids += [self.pad_token_id] * pad_len
            attention_mask += [0] * pad_len
            label_ids += [-100] * pad_len

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(label_ids, dtype=torch.long),
        }


def create_dataloader(sentences, tags, batch_size, tokenizer, max_len=64, shuffle=True):
    dataset = PhoBERTDataset(
        sentences=sentences,
        tags=tags,
        tokenizer=tokenizer,
        max_len=max_len
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle
    )

In [10]:
batch_size = 64
train_dataloader = create_dataloader(*prepare_data(train_path), batch_size=batch_size, tokenizer = tokenizer, max_len=64)
dev_dataloader = create_dataloader(*prepare_data(dev_path), batch_size=batch_size, tokenizer = tokenizer, max_len=64, shuffle=False)
test_dataloader = create_dataloader(*prepare_data(test_path), batch_size=batch_size, tokenizer = tokenizer, max_len=64)

# Create Model

In [11]:
def calculate_f1(preds, y):
    max_preds = preds.argmax(dim = 1, keepdim = True) # get the index of the max probability
    return f1_score(y.cpu(), max_preds.cpu(), average='macro')

In [12]:
class MultiTaskModel(nn.Module):
    def __init__(self, input_model, num_labels=3):
        super(MultiTaskModel, self).__init__()

        self.bert = input_model
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(
            self.bert.config.hidden_size,
            num_labels
        )

    def forward(self, input_ids, attention_mask):
        output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        last_hidden_state = output.last_hidden_state
        last_hidden_state = self.dropout(last_hidden_state)

        logits = self.classifier(last_hidden_state)

        return logits

In [13]:
from sklearn.metrics import f1_score
from tqdm import tqdm

def train(
    model,
    train_dataloader,
    dev_dataloader,
    criterion_span,
    optimizer_spans,
    device,
    num_epochs
):
    model.to(device)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0

        print("Epoch:", epoch + 1)

        for batch in tqdm(train_dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer_spans.zero_grad(set_to_none=True)

            logits = model(input_ids, attention_mask)

            loss = criterion_span(
                logits.view(-1, logits.shape[-1]),
                labels.view(-1)
            )

            loss.backward()
            optimizer_spans.step()

            total_loss += loss.item()

            del input_ids, attention_mask, labels, logits, loss

        train_loss = total_loss / len(train_dataloader)

        model.eval()
        val_loss = 0.0

        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in tqdm(dev_dataloader):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                logits = model(input_ids, attention_mask)

                loss = criterion_span(
                    logits.view(-1, logits.shape[-1]),
                    labels.view(-1)
                )

                val_loss += loss.item()

                preds = torch.argmax(logits, dim=-1)
                mask = labels != -100

                all_preds.extend(preds[mask].cpu().numpy().tolist())
                all_labels.extend(labels[mask].cpu().numpy().tolist())

                del input_ids, attention_mask, labels, logits, loss, preds, mask

        val_loss = val_loss / len(dev_dataloader)
        val_f1 = f1_score(all_labels, all_preds, average="macro")

        print("Training loss:", train_loss)
        print("Validation loss:", val_loss)
        print("Validation macro F1:", val_f1)

        torch.cuda.empty_cache()

# Start Training

In [14]:
import torch.optim as optim

num_epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultiTaskModel(
    input_model=input_model,
    num_labels=3
)

model.to(device)

criterion_span = nn.CrossEntropyLoss(ignore_index=-100)

optimizer_spans = optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=1e-5,
    foreach=False
)

train(
    model=model,
    train_dataloader=train_dataloader,
    dev_dataloader=dev_dataloader,
    criterion_span=criterion_span,
    optimizer_spans=optimizer_spans,
    device=device,
    num_epochs=num_epochs
)

Epoch: 1


100%|██████████| 18/18 [00:12<00:00,  1.48it/s]


Training loss: 0.5460266623136808
Validation loss: 0.4055961337354448
Validation macro F1: 0.5209226339082004
Epoch: 2


100%|██████████| 18/18 [00:12<00:00,  1.48it/s]


Training loss: 0.3546396274146416
Validation loss: 0.34028296917676926
Validation macro F1: 0.6698581663295707
Epoch: 3


...

Training loss: 0.10938931119849356
Validation loss: 0.4884723888503181
Validation macro F1: 0.7025816999030449
Epoch: 9


100%|██████████| 18/18 [00:12<00:00,  1.48it/s]


Training loss: 0.08789978109353738
Validation loss: 0.5168289807107713
Validation macro F1: 0.6957818497223451
Epoch: 10


100%|██████████| 18/18 [00:12<00:00,  1.48it/s]

Training loss: 0.07087382837701187
Validation loss: 0.5515174385574129
Validation macro F1: 0.7088392550133703


In [15]:
torch.save(model.state_dict(), f"/kaggle/working/phobert_large.pt")

# Load and test

In [16]:
import ast
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm


def parse_gold_spans(value):
    if pd.isna(value):
        return set()

    if isinstance(value, str):
        value = value.strip()
        if value == "" or value == "[]":
            return set()

        try:
            value = ast.literal_eval(value)
        except Exception:
            return set()

    result = set()

    def collect(x):
        if isinstance(x, int):
            result.add(x)
        elif isinstance(x, float) and not np.isnan(x):
            result.add(int(x))
        elif isinstance(x, (list, tuple, set)):
            for item in x:
                collect(item)

    collect(value)
    return result


def get_text_and_span_columns(df):
    possible_text_cols = ["content", "text", "comment", "sentence"]
    possible_span_cols = ["index_spans", "span_ids", "spans", "labels"]

    text_col = None
    span_col = None

    for col in possible_text_cols:
        if col in df.columns:
            text_col = col
            break

    for col in possible_span_cols:
        if col in df.columns:
            span_col = col
            break

    if text_col is None or span_col is None:
        raise ValueError(
            f"Không tìm được cột text/span. Columns hiện có: {df.columns.tolist()}"
        )

    return text_col, span_col


def load_bio_sentences(file_path):
    df = pd.read_csv(file_path)
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
    df.columns = [c.strip().lower() for c in df.columns]

    if "sentence_id" in df.columns:
        sentences = []
        for _, group in df.groupby("sentence_id", sort=False):
            words = group["word"].astype(str).tolist()
            sentences.append(words)
        return sentences

    sentences = []
    cur_words = []

    for _, row in df.iterrows():
        idx = row["index"]

        if idx == 0 and len(cur_words) > 0:
            sentences.append(cur_words)
            cur_words = []

        cur_words.append(str(row["word"]))

    if len(cur_words) > 0:
        sentences.append(cur_words)

    return sentences


def align_word_token_to_text(token, text, cursor):
    """
    PhoBERT Word BIO có thể có token dạng 'ngu_ngốc',
    còn text gốc là 'ngu ngốc'.
    Hàm này cố align cả token gốc và token thay '_' bằng space.
    """
    token = str(token)

    candidates = [
        token,
        token.replace("_", " "),
        token.replace("_", ""),
    ]

    for cand in candidates:
        start = text.find(cand, cursor)
        if start != -1:
            end = start + len(cand)
            return start, end

    # fallback: tìm từ đầu nếu cursor bị lệch
    for cand in candidates:
        start = text.find(cand)
        if start != -1:
            end = start + len(cand)
            return start, end

    return None


def align_tokens_to_text(tokens, text):
    offsets = []
    cursor = 0

    for tok in tokens:
        offset = align_word_token_to_text(tok, text, cursor)

        if offset is None:
            offsets.append(None)
            continue

        start, end = offset
        offsets.append((start, end))
        cursor = end

    return offsets


def pred_labels_to_char_indices(tokens, pred_labels, text):
    offsets = align_tokens_to_text(tokens, text)
    pred_chars = set()

    for label, offset in zip(pred_labels, offsets):
        if offset is None:
            continue

        if label != 0:
            start, end = offset
            pred_chars.update(range(start, end))

    return pred_chars


def count_contiguous_spans(char_indices):
    if not char_indices:
        return 0

    sorted_idx = sorted(char_indices)
    count = 1

    for i in range(1, len(sorted_idx)):
        if sorted_idx[i] != sorted_idx[i - 1] + 1:
            count += 1

    return count


def prf_from_char_sets(gold_sets, pred_sets):
    tp = 0
    pred_total = 0
    gold_total = 0

    for gold, pred in zip(gold_sets, pred_sets):
        tp += len(gold & pred)
        pred_total += len(pred)
        gold_total += len(gold)

    precision = tp / pred_total if pred_total > 0 else 0.0
    recall = tp / gold_total if gold_total > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return precision, recall, f1


def predict_bio_labels_phobert(model, tokenizer, sentences, device, max_len=64):
    model.eval()
    all_sentence_preds = []

    with torch.no_grad():
        for words in tqdm(sentences):
            input_ids = []
            token_word_map = []

            input_ids.append(tokenizer.cls_token_id)
            token_word_map.append(None)

            for word_idx, word in enumerate(words):
                sub_tokens = tokenizer.tokenize(str(word))
                sub_ids = tokenizer.convert_tokens_to_ids(sub_tokens)

                for j, sub_id in enumerate(sub_ids):
                    input_ids.append(sub_id)

                    # chỉ lấy prediction ở subword đầu tiên
                    if j == 0:
                        token_word_map.append(word_idx)
                    else:
                        token_word_map.append(None)

            input_ids.append(tokenizer.sep_token_id)
            token_word_map.append(None)

            input_ids = input_ids[:max_len]
            token_word_map = token_word_map[:max_len]

            attention_mask = [1] * len(input_ids)

            pad_len = max_len - len(input_ids)
            if pad_len > 0:
                input_ids += [tokenizer.pad_token_id] * pad_len
                attention_mask += [0] * pad_len
                token_word_map += [None] * pad_len

            input_ids_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)
            attention_mask_tensor = torch.tensor([attention_mask], dtype=torch.long).to(device)

            logits = model(input_ids_tensor, attention_mask_tensor)
            preds = torch.argmax(logits, dim=-1).squeeze(0).cpu().numpy().tolist()

            word_preds = [0] * len(words)

            for pred, word_idx in zip(preds, token_word_map):
                if word_idx is not None and word_idx < len(words):
                    word_preds[word_idx] = pred

            all_sentence_preds.append(word_preds)

    return all_sentence_preds


def evaluate_span_level_official_phobert(
    model,
    tokenizer,
    test_bio_path,
    test_gold_path,
    device,
    model_name="vinai/phobert-large",
    max_len=64,
    save_csv=True
):
    test_sentences = load_bio_sentences(test_bio_path)

    gold_df = pd.read_csv(test_gold_path)
    gold_df = gold_df.loc[:, ~gold_df.columns.str.contains("^Unnamed")]
    gold_df.columns = [c.strip() for c in gold_df.columns]

    text_col, span_col = get_text_and_span_columns(gold_df)

    gold_texts = gold_df[text_col].astype(str).tolist()
    gold_sets = gold_df[span_col].apply(parse_gold_spans).tolist()

    if len(test_sentences) != len(gold_texts):
        print("WARNING: Số câu BIO và số dòng gold test.csv không khớp!")
        print("BIO sentences:", len(test_sentences))
        print("Gold rows:", len(gold_texts))

        n = min(len(test_sentences), len(gold_texts))
        test_sentences = test_sentences[:n]
        gold_texts = gold_texts[:n]
        gold_sets = gold_sets[:n]

    pred_label_sentences = predict_bio_labels_phobert(
        model=model,
        tokenizer=tokenizer,
        sentences=test_sentences,
        device=device,
        max_len=max_len
    )

    pred_sets = []

    for tokens, pred_labels, text in zip(test_sentences, pred_label_sentences, gold_texts):
        pred_chars = pred_labels_to_char_indices(
            tokens=tokens,
            pred_labels=pred_labels,
            text=text
        )
        pred_sets.append(pred_chars)

    single_indices = [
        i for i, gold in enumerate(gold_sets)
        if count_contiguous_spans(gold) == 1
    ]

    multiple_indices = [
        i for i, gold in enumerate(gold_sets)
        if count_contiguous_spans(gold) > 1
    ]

    all_indices = [
        i for i, gold in enumerate(gold_sets)
        if count_contiguous_spans(gold) >= 1
    ]

    def subset_score(indices):
        subset_gold = [gold_sets[i] for i in indices]
        subset_pred = [pred_sets[i] for i in indices]
        return prf_from_char_sets(subset_gold, subset_pred)

    single_p, single_r, single_f1 = subset_score(single_indices)
    multiple_p, multiple_r, multiple_f1 = subset_score(multiple_indices)
    all_p, all_r, all_f1 = subset_score(all_indices)

    result = {
        "model": model_name,

        "single_precision": single_p,
        "single_recall": single_r,
        "single_f1": single_f1,

        "multiple_precision": multiple_p,
        "multiple_recall": multiple_r,
        "multiple_f1": multiple_f1,

        "all_precision": all_p,
        "all_recall": all_r,
        "all_f1": all_f1,

        "n_single": len(single_indices),
        "n_multiple": len(multiple_indices),
        "n_all": len(all_indices),
    }

    print("===== PhoBERT span-level official-style benchmark =====")
    for k, v in result.items():
        if k == "model":
            print(f"{k}: {v}")
        elif k.startswith("n_"):
            print(f"{k}: {v}")
        else:
            print(f"{k}: {v:.6f}")

    if save_csv:
        result_path = Path("/kaggle/working/phobert_large_span_level_results.csv")

        if result_path.exists():
            old_df = pd.read_csv(result_path)
            new_df = pd.concat([old_df, pd.DataFrame([result])], ignore_index=True)
        else:
            new_df = pd.DataFrame([result])

        new_df.to_csv(result_path, index=False)
        print(f"\nSaved result to: {result_path}")

    return result, pred_sets, gold_sets

In [17]:
model = MultiTaskModel(input_model = input_model)
model.load_state_dict(torch.load("/kaggle/working/phobert_large.pt"))
model.to(device)

span_result_base, pred_spans_base, gold_spans = evaluate_span_level_official_phobert(
    model=model,
    tokenizer=tokenizer,
    test_bio_path=test_bio_path,
    test_gold_path=test_gold_path,
    device=device,
    model_name="vinai/phobert-large",
    max_len=64,
    save_csv=True
)

100%|██████████| 1106/1106 [00:21<00:00, 51.33it/s]

===== PhoBERT span-level official-style benchmark =====
model: vinai/phobert-large
single_precision: 0.729260
single_recall: 0.511733
single_f1: 0.601432
multiple_precision: 0.764132
multiple_recall: 0.644713
multiple_f1: 0.699361
all_precision: 0.756901
all_recall: 0.612893
all_f1: 0.677327
n_single: 235
n_multiple: 296
n_all: 531

Saved result to: /kaggle/working/phobert_large_span_level_results.csv
